# Pathogen & Disease NER — Tick & Climate Papers
### SciSpacy `en_ner_bc5cdr_md` (faithful) **+ tick-borne-disease gazetteer** (extension)

Ports the phage paper's disease-entity module to tick *papers* and extends it for the tick domain.

**Faithful core** (as in the paper): SciSpacy `en_ner_bc5cdr_md` extracts `DISEASE` entities from a
punctuation-stripped, case-varied, concatenated title+abstract corpus; results land in a `Diseases`
column; names are refined (OpenRefine step) and standardised against **ICD-11**.

**Extensions for ticks:**
1. A curated **tick-borne-disease + tick-vector gazetteer** (spaCy `PhraseMatcher`) that catches domain
   terms the general biomedical model misses — babesiosis, theileriosis/East Coast fever, CCHF, TBE,
   African tick-bite fever, heartwater, *Ixodes*, *Amblyomma*, *Rhipicephalus*, *Hyalomma*, etc. The list is
   deliberately **SSA- and One-Health-aware**, since that serves your Sub-Saharan-Africa-share and
   climate-niche findings.
2. Programmatic **normalisation** (surface form → canonical disease) — the scriptable OpenRefine equivalent,
   plus an export you can still cluster in the OpenRefine GUI if you want the faithful manual pass.
3. An **ICD-11 mapping** with three axes: ICD-11 *chapter* (faithful, mostly "01 Certain infectious or
   parasitic diseases"), a **pathogen class** (spirochaete / rickettsial / other-bacterial / viral /
   protozoal), and a **host axis** (human / veterinary / zoonotic). Fine-grained ICD-11 **stem codes** are
   left for verification (only Lyme = `1C1G` is confirmed) with an optional WHO ICD-11 API cell to fill the rest.

**Outputs feed:** disease-frequency tables, **disease × year** (climate-niche trend) and **disease × country**
(SSA share) matrices — the two headline analyses for the paper.


## 1. Install
_SciSpacy + the BC5CDR disease model. The paper pinned model **v0.5.3**; on current Colab, install a scispacy that ships `en_ner_bc5cdr_md` and pin spaCy if you hit a version clash._

In [ ]:
!python -m pip uninstall -y numpy pandas scipy scikit-learn spacy thinc blis scispacy en-ner-bc5cdr-md

!python -m pip install --no-cache-dir --force-reinstall \
  numpy==1.26.4 \
  pandas==2.2.3 \
  scipy==1.13.1 \
  scikit-learn==1.5.2 \
  spacy==3.7.5 \
  thinc==8.2.5 \
  blis==0.7.11 \
  scispacy==0.5.5

!python -m pip install --no-cache-dir --no-deps --force-reinstall \
  https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

!python -m pip check

In [ ]:
import re, string, warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [ ]:
import numpy as np
import pandas as pd
import scipy
import sklearn
import spacy
import scispacy

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("spacy:", spacy.__version__)
print("scispacy:", scispacy.__version__)

nlp = spacy.load("en_ner_bc5cdr_md")
print("loaded:", nlp.pipe_names)

In [ ]:
text = "Aspirin can cause gastritis and is used in cardiovascular disease."
doc = nlp(text)

[(ent.text, ent.label_) for ent in doc.ents]

In [ ]:
# # scispacy + BC5CDR disease NER model
# !pip install -q scispacy
# !pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_ner_bc5cdr_md-0.5.3.tar.gz
# !pip install -q pandas pyarrow matplotlib seaborn
# # If import fails with a spaCy compatibility error, pin: !pip install -q "spacy>=3.6,<3.8"
# print("install step done")

In [ ]:


# nlp = spacy.load("en_ner_bc5cdr_md")   # faithful: BC5CDR disease/chemical NER
# print("loaded:", nlp.pipe_names)

## 2. CONFIG
_Same corpus and column mapping as Notebook 01._

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
INPUT_PARQUET = '/content/drive/MyDrive/COGNOSX/TICKs_and_climate_change/preprocessing_data_v2/master_tick_checked.parquet'

OUTPUT_DIR    = "/content/drive/MyDrive/COGNOSX/TICKs_and_climate_change/preprocessing_data_v2/ner_outputs"

COL_ID       = "record_id"
COL_TITLE    = "title"
COL_ABSTRACT = "abstract"
KEY_WORDS    = "keywords"
COL_YEAR     = "year"
COL_COUNTRY  = "oa_countries"     # string OR list/nested; set None if absent. Used for the SSA-share matrix.
COL_LANG     = "language"    # set None if absent

ENGLISH_ONLY = False         # BC5CDR is English-trained; True gives cleaner NER (see NB01 note)

# Faithful case-variation trick (paper concatenated multiple case forms before NER).
# True = faithful but ~2x slower and needs dedup; False = single normalised pass (recommended at 70k).
FAITHFUL_CASE_VARIATIONS = False

# scispacy throughput
BATCH_SIZE = 200
N_PROCESS  = 1               # raise on a multi-core runtime

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)
print("config set")

In [ ]:
COL_COUNTRY = 'oa_countries'

## 3. Load corpus (same filter as NB01)

In [ ]:
df = pd.read_parquet(INPUT_PARQUET)

def _s(x):
    return "" if (x is None or (isinstance(x, float) and np.isnan(x))) else str(x)

work = df.copy()
has_title = work[COL_TITLE].map(lambda x: len(_s(x).strip()) > 0)
has_abs   = work[COL_ABSTRACT].map(lambda x: len(_s(x).strip()) > 0)
has_keyw = work[KEY_WORDS].map(lambda x: len(_s(x).strip()) > 0)
work = work[has_title & has_abs  & has_keyw].copy()
if ENGLISH_ONLY and COL_LANG:
    work = work[work[COL_LANG].astype(str).str.lower().str.startswith("en")].copy()

work["_text"] = (work[COL_TITLE].map(_s) + ". " + work[COL_ABSTRACT].map(_s) + ". " + work[KEY_WORDS].map(_s)).str.strip()
work[COL_YEAR] = pd.to_numeric(work[COL_YEAR], errors="coerce")
print("records for NER:", len(work))

## 4. Faithful preprocessing for NER
Remove punctuation, then (optionally) build the paper's concatenated case variations.

In [ ]:
import string
_punct = str.maketrans("", "", string.punctuation)

def strip_punct(t):
    return _s(t).translate(_punct)

def ner_input(text):
    base = strip_punct(text)
    if FAITHFUL_CASE_VARIATIONS:
        # paper concatenated multiple case forms to hedge NER case-sensitivity
        return base + " " + base.lower()
    return base

work["_ner_text"] = work["_text"].map(ner_input)
print(work["_ner_text"].iloc[0][:200])

## 5. Faithful disease extraction (`extract_diseases`)
The BC5CDR model tags `DISEASE` and `CHEMICAL`; we keep `DISEASE`, matching the paper.

In [ ]:
COL_ID       = "record_id"

In [ ]:
def extract_diseases_doc(doc):
    out = []
    for ent in doc.ents:
        if ent.label_ == "DISEASE":
            out.append(ent.text.strip().lower())
    return out

texts = work["_ner_text"].tolist()
disease_lists = []
for doc in nlp.pipe(texts, batch_size=BATCH_SIZE, n_process=N_PROCESS):
    disease_lists.append(sorted(set(extract_diseases_doc(doc))))   # dedup within record

work["Diseases"] = disease_lists      # faithful column name
print("records with >=1 disease entity:", int((work["Diseases"].map(len) > 0).sum()))
work[[COL_ID, "Diseases"]].head()

In [ ]:
work[[COL_ID, "Diseases"]].head()

In [ ]:
disease_lists

## 6. EXTENSION 1 — tick-borne-disease & vector gazetteer
`PhraseMatcher` over raw title+abstract (not punctuation-stripped, so multi-word species names survive).
Catches domain entities BC5CDR misses. Curated for human + veterinary + SSA relevance.

In [ ]:
from spacy.matcher import PhraseMatcher

# canonical disease -> surface forms (extend freely)
TBD_TERMS = {
    "Lyme borreliosis": ["lyme", "lyme disease", "lyme borreliosis", "borreliosis", "borrelia burgdorferi", "neuroborreliosis"],
    "Tick-borne relapsing fever": ["relapsing fever", "borrelia recurrentis", "borrelia duttonii", "borrelia hermsii"],
    "Rocky Mountain spotted fever": ["rocky mountain spotted fever", "rickettsia rickettsii"],
    "African tick-bite fever": ["african tick bite fever", "african tick-bite fever", "rickettsia africae"],
    "Mediterranean spotted fever": ["mediterranean spotted fever", "boutonneuse fever", "rickettsia conorii"],
    "Spotted fever rickettsiosis": ["spotted fever", "spotted fever group", "rickettsiosis", "tick typhus"],
    "Anaplasmosis": ["anaplasmosis", "anaplasma phagocytophilum", "anaplasma marginale", "human granulocytic anaplasmosis"],
    "Ehrlichiosis": ["ehrlichiosis", "ehrlichia chaffeensis", "human monocytic ehrlichiosis"],
    "Heartwater": ["heartwater", "cowdriosis", "ehrlichia ruminantium"],
    "Q fever": ["q fever", "coxiella burnetii"],
    "Tularemia": ["tularemia", "tularaemia", "francisella tularensis"],
    "Bartonellosis": ["bartonellosis", "bartonella"],
    "Tick-borne encephalitis": ["tick borne encephalitis", "tick-borne encephalitis", "tbe", "tbev"],
    "Crimean-Congo haemorrhagic fever": ["crimean congo haemorrhagic fever", "crimean-congo hemorrhagic fever", "cchf", "cchfv"],
    "Kyasanur Forest disease": ["kyasanur forest disease", "kfd"],
    "Omsk haemorrhagic fever": ["omsk haemorrhagic fever", "omsk hemorrhagic fever"],
    "Powassan virus disease": ["powassan"],
    "Colorado tick fever": ["colorado tick fever"],
    "Alkhurma haemorrhagic fever": ["alkhurma", "alkhumra"],
    "Severe fever with thrombocytopenia syndrome": ["severe fever with thrombocytopenia syndrome", "sfts", "sftsv"],
    "Louping ill": ["louping ill"],
    "Babesiosis": ["babesiosis", "babesia", "babesia microti", "babesia bovis", "babesia bigemina", "redwater"],
    "Theileriosis / East Coast fever": ["theileriosis", "theileria", "east coast fever", "theileria parva", "theileria annulata", "tropical theileriosis"],
    "Cytauxzoonosis": ["cytauxzoonosis", "cytauxzoon"],
    "Alpha-gal syndrome": ["alpha gal", "alpha-gal", "mammalian meat allergy"],
    "Tick paralysis": ["tick paralysis"],
}

# tick vector genera/species (One-Health axis; not diseases, tracked separately)
TICK_VECTORS = {
    "Ixodes": ["ixodes", "ixodes ricinus", "ixodes scapularis", "ixodes persulcatus", "ixodes pacificus"],
    "Amblyomma": ["amblyomma", "amblyomma americanum", "amblyomma variegatum", "amblyomma hebraeum"],
    "Rhipicephalus": ["rhipicephalus", "rhipicephalus sanguineus", "rhipicephalus microplus", "rhipicephalus appendiculatus", "boophilus"],
    "Dermacentor": ["dermacentor", "dermacentor variabilis", "dermacentor andersoni", "dermacentor reticulatus", "dermacentor marginatus"],
    "Hyalomma": ["hyalomma", "hyalomma marginatum", "hyalomma rufipes"],
    "Haemaphysalis": ["haemaphysalis"],
    "Ornithodoros": ["ornithodoros"],
}

def build_matcher(term_map):
    m = PhraseMatcher(nlp.vocab, attr="LOWER")
    lookup = {}
    for canonical, forms in term_map.items():
        patterns = [nlp.make_doc(f) for f in forms]
        key = canonical.replace(" ", "_")
        m.add(key, patterns)
        lookup[key] = canonical
    return m, lookup

tbd_matcher, tbd_lookup = build_matcher(TBD_TERMS)
vec_matcher, vec_lookup = build_matcher(TICK_VECTORS)

def match_canonical(text, matcher, lookup):
    doc = nlp.make_doc(_s(text))
    found = set()
    for mid, s, e in matcher(doc):
        found.add(lookup[nlp.vocab.strings[mid]])
    return sorted(found)

work["TBD_gazetteer"] = work["_text"].map(lambda t: match_canonical(t, tbd_matcher, tbd_lookup))
work["Tick_vectors"]  = work["_text"].map(lambda t: match_canonical(t, vec_matcher, vec_lookup))

print("records matching a TBD term:", int((work["TBD_gazetteer"].map(len) > 0).sum()))
print("records naming a tick vector:", int((work["Tick_vectors"].map(len) > 0).sum()))
work[[COL_ID, "TBD_gazetteer", "Tick_vectors"]].head()

## 7. EXTENSION 2 — normalise the free NER output (scriptable OpenRefine)
Map BC5CDR surface forms onto canonical names where they overlap the gazetteer, and export the
remaining long tail for optional manual clustering in the OpenRefine GUI (the faithful step).

In [ ]:
# reverse index: surface form -> canonical
SURFACE2CANON = {}
for canon, forms in TBD_TERMS.items():
    for f in forms:
        SURFACE2CANON[f.lower()] = canon

def normalise_bc5cdr(disease_list):
    canon = set()
    for d in disease_list:
        d0 = d.lower().strip()
        if d0 in SURFACE2CANON:
            canon.add(SURFACE2CANON[d0])
    return sorted(canon)

work["Diseases_TBD_from_bc5cdr"] = work["Diseases"].map(normalise_bc5cdr)

# union: canonical TBDs from EITHER the statistical model or the gazetteer
def union_lists(a, b):
    return sorted(set(a) | set(b))
work["Diseases_canonical"] = [union_lists(a, b) for a, b in
                              zip(work["Diseases_TBD_from_bc5cdr"], work["TBD_gazetteer"])]

# export the raw BC5CDR long tail (non-TBD terms) for OpenRefine clustering if desired
from collections import Counter
tail = Counter()
for lst in work["Diseases"]:
    for d in lst:
        if d.lower() not in SURFACE2CANON:
            tail[d] += 1
tail_df = pd.DataFrame(tail.most_common(), columns=["surface_form", "count"])
tail_df.to_csv(f"{OUTPUT_DIR}/bc5cdr_disease_longtail_for_openrefine.csv", index=False)
print("long-tail terms exported for OpenRefine:", len(tail_df))
tail_df.head(20)

## 8. EXTENSION 3 — ICD-11 mapping (chapter + pathogen class + host)
Faithful chapter mapping (almost all tick-borne infections sit in **Chapter 01**). Pathogen-class and host
axes are the scientifically useful ones for a tick paper. **Only Lyme's stem code (`1C1G`) is verified** —
the rest are left blank for verification (see the WHO API cell).

In [ ]:
# canonical disease -> (icd11_chapter, icd11_stem_VERIFY, pathogen_class, host_axis)
# icd11_stem: "" means VERIFY at https://icd.who.int/browse11 or via the API cell below.
ICD11_MAP = {
    "Lyme borreliosis":               ("01", "1C1G", "spirochaete",   "human"),
    "Tick-borne relapsing fever":     ("01", "",     "spirochaete",   "human"),
    "Rocky Mountain spotted fever":   ("01", "",     "rickettsial",   "human"),
    "African tick-bite fever":        ("01", "",     "rickettsial",   "human"),
    "Mediterranean spotted fever":    ("01", "",     "rickettsial",   "human"),
    "Spotted fever rickettsiosis":    ("01", "",     "rickettsial",   "human"),
    "Anaplasmosis":                   ("01", "",     "rickettsial",   "zoonotic"),
    "Ehrlichiosis":                   ("01", "",     "rickettsial",   "zoonotic"),
    "Heartwater":                     ("01", "",     "rickettsial",   "veterinary"),
    "Q fever":                        ("01", "",     "other-bacterial","zoonotic"),
    "Tularemia":                      ("01", "",     "other-bacterial","zoonotic"),
    "Bartonellosis":                  ("01", "",     "other-bacterial","zoonotic"),
    "Tick-borne encephalitis":        ("01", "",     "viral",         "human"),
    "Crimean-Congo haemorrhagic fever":("01","",     "viral",         "zoonotic"),
    "Kyasanur Forest disease":        ("01", "",     "viral",         "zoonotic"),
    "Omsk haemorrhagic fever":        ("01", "",     "viral",         "zoonotic"),
    "Powassan virus disease":         ("01", "",     "viral",         "human"),
    "Colorado tick fever":            ("01", "",     "viral",         "human"),
    "Alkhurma haemorrhagic fever":    ("01", "",     "viral",         "zoonotic"),
    "Severe fever with thrombocytopenia syndrome": ("01","","viral",  "zoonotic"),
    "Louping ill":                    ("01", "",     "viral",         "veterinary"),
    "Babesiosis":                     ("01", "",     "protozoal",     "zoonotic"),
    "Theileriosis / East Coast fever":("01", "",     "protozoal",     "veterinary"),
    "Cytauxzoonosis":                 ("01", "",     "protozoal",     "veterinary"),
    "Alpha-gal syndrome":             ("04", "",     "non-infectious","human"),   # immune-mediated
    "Tick paralysis":                 ("08", "",     "toxin-mediated","human"),
}
icd_df = pd.DataFrame(
    [(k, *v) for k, v in ICD11_MAP.items()],
    columns=["disease", "icd11_chapter", "icd11_stem", "pathogen_class", "host_axis"]
)
icd_df.to_csv(f"{OUTPUT_DIR}/tbd_icd11_map.csv", index=False)
icd_df

### (Optional) Fill stem codes via the WHO ICD-11 API
The rigorous route for publication. Register a free client at
https://icd.who.int/icdapi to get `CLIENT_ID`/`CLIENT_SECRET`, then this cell queries the MMS linearization
and returns the best-matching stem code for each canonical disease. Left off by default.

In [ ]:
RUN_ICD11_API = False   # set True after adding credentials
CLIENT_ID = ""          # from https://icd.who.int/icdapi
CLIENT_SECRET = ""

if RUN_ICD11_API:
    import requests
    tok = requests.post("https://icdaccessmanagement.who.int/connect/token",
                        data={"client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
                              "scope": "icdapi_access", "grant_type": "client_credentials"}).json()
    headers = {"Authorization": f"Bearer {tok['access_token']}",
               "Accept": "application/json", "Accept-Language": "en",
               "API-Version": "v2"}
    def icd11_lookup(name):
        r = requests.get("https://id.who.int/icd/release/11/2024-01/mms/search",
                         params={"q": name, "flatResults": "true"}, headers=headers)
        ents = r.json().get("destinationEntities", [])
        if not ents:
            return ("", "")
        top = ents[0]
        return (top.get("theCode", ""), re.sub("<[^<]+?>", "", top.get("title", "")))
    codes = {d: icd11_lookup(d) for d in ICD11_MAP}
    for d, (code_, title_) in codes.items():
        print(f"{d:45s} -> {code_:8s} {title_}")
else:
    print("ICD-11 API disabled — using verified/curated map above.")

## 9. Aggregations for the paper

In [ ]:
# explode canonical diseases
exp = work.explode("Diseases_canonical").dropna(subset=["Diseases_canonical"])
exp = exp[exp["Diseases_canonical"] != ""]

# 9a. disease frequency
freq = exp["Diseases_canonical"].value_counts().rename_axis("disease").reset_index(name="papers")
freq = freq.merge(icd_df[["disease","pathogen_class","host_axis"]], on="disease", how="left")
freq.to_csv(f"{OUTPUT_DIR}/disease_frequency.csv", index=False)
display(freq.head(20))

plt.figure(figsize=(9,6))
top = freq.head(20).iloc[::-1]
plt.barh(top["disease"], top["papers"])
plt.xlabel("Papers"); plt.title("Top tick-borne diseases by paper count"); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/disease_frequency.png", dpi=150); plt.show()

### 9b. Disease × year — climate-niche trend signal

In [ ]:
ey = exp.dropna(subset=[COL_YEAR]).copy()
ey[COL_YEAR] = ey[COL_YEAR].astype(int)
dy = (ey.groupby([COL_YEAR, "Diseases_canonical"]).size()
        .reset_index(name="papers")
        .pivot(index=COL_YEAR, columns="Diseases_canonical", values="papers").fillna(0))
dy.to_csv(f"{OUTPUT_DIR}/disease_by_year.csv")

plt.figure(figsize=(11,6))
for d in freq.head(8)["disease"]:
    if d in dy.columns:
        plt.plot(dy.index, dy[d], label=d)
plt.xlabel("Publication year"); plt.ylabel("Papers")
plt.title("Temporal trend of leading tick-borne diseases")
plt.legend(fontsize=8, ncol=2); plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/disease_by_year.png", dpi=150); plt.show()

### 9c. Country extraction — presence-based, deduped (counting rule)
**Rule:** a paper counts **once** for country X if *any* of its authors is affiliated in X. So we take the
**set** of distinct countries per paper before counting — three US co-authors still count the US once.
`to_countries` handles a flat string, a list of countries, or a list of affiliation strings, and normalises
common bibliographic name variants. This same helper should be reused in the network notebook so the SSA
number is identical across every figure.

In [ ]:
# common affiliation-metadata variants -> canonical country name
COUNTRY_ALIASES = {
    "usa": "United States", "u.s.a.": "United States", "us": "United States",
    "united states of america": "United States", "united states": "United States",
    "uk": "United Kingdom", "u.k.": "United Kingdom", "england": "United Kingdom",
    "scotland": "United Kingdom", "wales": "United Kingdom", "great britain": "United Kingdom",
    "russian federation": "Russia", "korea": "South Korea",
    "republic of korea": "South Korea", "korea, rep.": "South Korea",
    "ivory coast": "Cote d'Ivoire", "cote d ivoire": "Cote d'Ivoire", "côte d’ivoire": "Cote d'Ivoire",
    "democratic republic of the congo": "DR Congo", "dr congo": "DR Congo",
    "congo, dem. rep.": "DR Congo", "drc": "DR Congo", "congo-kinshasa": "DR Congo",
    "republic of the congo": "Congo", "congo, rep.": "Congo", "congo-brazzaville": "Congo",
    "united republic of tanzania": "Tanzania", "tanzania, united rep.": "Tanzania",
    "swaziland": "Eswatini", "cape verde": "Cabo Verde", "the gambia": "Gambia",
    "czech republic": "Czechia", "iran (islamic republic of)": "Iran",

    # ---- ISO-2 codes (lower-case keys) ----
    "ad":"Andorra","ae":"United Arab Emirates","af":"Afghanistan",
    "ag":"Antigua and Barbuda","al":"Albania","am":"Armenia","ao":"Angola",
    "ar":"Argentina","at":"Austria","au":"Australia","az":"Azerbaijan",
    "ba":"Bosnia and Herzegovina","bb":"Barbados","bd":"Bangladesh","be":"Belgium",
    "bf":"Burkina Faso","bg":"Bulgaria","bh":"Bahrain","bi":"Burundi","bj":"Benin",
    "bo":"Bolivia","br":"Brazil","bt":"Bhutan","bw":"Botswana","by":"Belarus",
    "bz":"Belize","ca":"Canada","cd":"DR Congo","ch":"Switzerland",
    "ci":"Cote d'Ivoire","cl":"Chile","cm":"Cameroon","cn":"China","co":"Colombia",
    "cr":"Costa Rica","cu":"Cuba","cv":"Cabo Verde","cy":"Cyprus","cz":"Czechia",
    "de":"Germany","dj":"Djibouti","dk":"Denmark","dz":"Algeria","ec":"Ecuador",
    "ee":"Estonia","eg":"Egypt","es":"Spain","et":"Ethiopia","fi":"Finland",
    "fj":"Fiji","fr":"France","ga":"Gabon","gb":"United Kingdom","ge":"Georgia",
    "gh":"Ghana","gm":"Gambia","gn":"Guinea","gr":"Greece","gt":"Guatemala",
    "gw":"Guinea-Bissau","hk":"Hong Kong","hn":"Honduras","hr":"Croatia","ht":"Haiti",
    "hu":"Hungary","id":"Indonesia","ie":"Ireland","il":"Israel","in":"India",
    "iq":"Iraq","ir":"Iran","is":"Iceland","it":"Italy","jm":"Jamaica","jo":"Jordan",
    "jp":"Japan","ke":"Kenya","kg":"Kyrgyzstan","kh":"Cambodia","kp":"North Korea",
    "kr":"South Korea","kw":"Kuwait","kz":"Kazakhstan","la":"Laos","lb":"Lebanon",
    "li":"Liechtenstein","lk":"Sri Lanka","lr":"Liberia","ls":"Lesotho","lt":"Lithuania",
    "lu":"Luxembourg","lv":"Latvia","ly":"Libya","ma":"Morocco","md":"Moldova",
    "me":"Montenegro","mg":"Madagascar","mk":"North Macedonia","ml":"Mali","mm":"Myanmar",
    "mn":"Mongolia","mr":"Mauritania","mt":"Malta","mu":"Mauritius","mw":"Malawi",
    "mx":"Mexico","my":"Malaysia","mz":"Mozambique","ne":"Niger","ng":"Nigeria",
    "ni":"Nicaragua","nl":"Netherlands","no":"Norway","np":"Nepal","nz":"New Zealand",
    "om":"Oman","pa":"Panama","pe":"Peru","pg":"Papua New Guinea","ph":"Philippines",
    "pk":"Pakistan","pl":"Poland","ps":"Palestine","pt":"Portugal","py":"Paraguay",
    "qa":"Qatar","ro":"Romania","rs":"Serbia","ru":"Russia","rw":"Rwanda",
    "sa":"Saudi Arabia","sd":"Sudan","se":"Sweden","sg":"Singapore","si":"Slovenia",
    "sk":"Slovakia","sl":"Sierra Leone","sn":"Senegal","so":"Somalia","ss":"South Sudan",
    "sv":"El Salvador","sy":"Syria","sz":"Eswatini","td":"Chad","tg":"Togo",
    "th":"Thailand","tj":"Tajikistan","tn":"Tunisia","tr":"Turkey",
    "tt":"Trinidad and Tobago","tw":"Taiwan","tz":"Tanzania","ua":"Ukraine",
    "ug":"Uganda","us":"United States","uy":"Uruguay","uz":"Uzbekistan",
    "ve":"Venezuela","vn":"Vietnam","xk":"Kosovo","ye":"Yemen","za":"South Africa",
    "zm":"Zambia","zw":"Zimbabwe",

    "AD": "Andorra",
    "AE": "United Arab Emirates",
    "AF": "Afghanistan",
    "AG": "Antigua and Barbuda",
    "AL": "Albania",
    "AM": "Armenia",
    "AO": "Angola",
    "AR": "Argentina",
    "AT": "Austria",
    "AU": "Australia",
    "AX": "Åland Islands",
    "AZ": "Azerbaijan",
    "BA": "Bosnia and Herzegovina",
    "BB": "Barbados",
    "BD": "Bangladesh",
    "BE": "Belgium",
    "BF": "Burkina Faso",
    "BG": "Bulgaria",
    "BH": "Bahrain",
    "BI": "Burundi",
    "BJ": "Benin",
    "BO": "Bolivia",
    "BR": "Brazil",
    "BT": "Bhutan",
    "BW": "Botswana",
    "BY": "Belarus",
    "BZ": "Belize",
    "CA": "Canada",
    "CD": "Democratic Republic of the Congo",
    "CH": "Switzerland",
    "CI": "Côte d'Ivoire",
    "CL": "Chile",
    "CM": "Cameroon",
    "CN": "China",
    "CO": "Colombia",
    "CR": "Costa Rica",
    "CU": "Cuba",
    "CV": "Cape Verde",
    "CW": "Curaçao",
    "CY": "Cyprus",
    "CZ": "Czech Republic",
    "DE": "Germany",
    "DJ": "Djibouti",
    "DK": "Denmark",
    "DZ": "Algeria",
    "EC": "Ecuador",
    "EE": "Estonia",
    "EG": "Egypt",
    "ES": "Spain",
    "ET": "Ethiopia",
    "FI": "Finland",
    "FJ": "Fiji",
    "FR": "France",
    "GA": "Gabon",
    "GB": "United Kingdom",
    "GD": "Grenada",
    "GE": "Georgia",
    "GF": "French Guiana",
    "GH": "Ghana",
    "GL": "Greenland",
    "GM": "Gambia",
    "GN": "Guinea",
    "GP": "Guadeloupe",
    "GR": "Greece",
    "GT": "Guatemala",
    "GW": "Guinea-Bissau",
    "HK": "Hong Kong",
    "HN": "Honduras",
    "HR": "Croatia",
    "HT": "Haiti",
    "HU": "Hungary",
    "ID": "Indonesia",
    "IE": "Ireland",
    "IL": "Israel",
    "IM": "Isle of Man",
    "IN": "India",
    "IQ": "Iraq",
    "IR": "Iran",
    "IS": "Iceland",
    "IT": "Italy",
    "JM": "Jamaica",
    "JO": "Jordan",
    "JP": "Japan",
    "KE": "Kenya",
    "KG": "Kyrgyzstan",
    "KH": "Cambodia",
    "KN": "Saint Kitts and Nevis",
    "KP": "North Korea",
    "KR": "South Korea",
    "KW": "Kuwait",
    "KZ": "Kazakhstan",
    "LA": "Laos",
    "LB": "Lebanon",
    "LI": "Liechtenstein",
    "LK": "Sri Lanka",
    "LR": "Liberia",
    "LS": "Lesotho",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "LY": "Libya",
    "MA": "Morocco",
    "MD": "Moldova",
    "ME": "Montenegro",
    "MG": "Madagascar",
    "MK": "North Macedonia",
    "ML": "Mali",
    "MM": "Myanmar",
    "MN": "Mongolia",
    "MO": "Macau",
    "MQ": "Martinique",
    "MR": "Mauritania",
    "MT": "Malta",
    "MU": "Mauritius",
    "MV": "Maldives",
    "MW": "Malawi",
    "MX": "Mexico",
    "MY": "Malaysia",
    "MZ": "Mozambique",
    "NC": "New Caledonia",
    "NE": "Niger",
    "NG": "Nigeria",
    "NI": "Nicaragua",
    "NL": "Netherlands",
    "NO": "Norway",
    "NP": "Nepal",
    "NZ": "New Zealand",
    "OM": "Oman",
    "PA": "Panama",
    "PE": "Peru",
    "PF": "French Polynesia",
    "PG": "Papua New Guinea",
    "PH": "Philippines",
    "PK": "Pakistan",
    "PL": "Poland",
    "PR": "Puerto Rico",
    "PS": "Palestine",
    "PT": "Portugal",
    "PY": "Paraguay",
    "QA": "Qatar",
    "RE": "Réunion",
    "RO": "Romania",
    "RS": "Serbia",
    "RU": "Russia",
    "RW": "Rwanda",
    "SA": "Saudi Arabia",
    "SD": "Sudan",
    "SE": "Sweden",
    "SG": "Singapore",
    "SI": "Slovenia",
    "SJ": "Svalbard and Jan Mayen",
    "SK": "Slovakia",
    "SL": "Sierra Leone",
    "SN": "Senegal",
    "SO": "Somalia",
    "SS": "South Sudan",
    "ST": "São Tomé and Príncipe",
    "SV": "El Salvador",
    "SX": "Sint Maarten",
    "SY": "Syria",
    "SZ": "Eswatini",
    "TD": "Chad",
    "TG": "Togo",
    "TH": "Thailand",
    "TJ": "Tajikistan",
    "TN": "Tunisia",
    "TO": "Tonga",
    "TR": "Turkey",
    "TT": "Trinidad and Tobago",
    "TW": "Taiwan",
    "TZ": "Tanzania",
    "UA": "Ukraine",
    "UG": "Uganda",
    "UY": "Uruguay",
    "UZ": "Uzbekistan",
    "US": "United States",
    "VE": "Venezuela",
    "VN": "Vietnam",
    "XK": "Kosovo",
    "YE": "Yemen",
    "ZA": "South Africa",
    "ZM": "Zambia",
    "ZW": "Zimbabwe",
    None: None,
    float("nan"): None
}

def canon_country(c):
    c = _s(c).strip()
    return COUNTRY_ALIASES.get(c, c) #c.lower(),

def to_countries(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        raw = list(x)
    elif _s(x):
        raw = re.split(r"[;|]", _s(x))
    else:
        raw = []
    # if entries look like full affiliation strings, take the trailing token as country
    cleaned = []
    for r in raw:
        r = _s(r).strip()
        if not r:
            continue
        if "," in r and len(r.split(",")[-1].strip()) > 1:
            r = r.split(",")[-1].strip()   # ".. , Nairobi, Kenya" -> "Kenya"
        cleaned.append(canon_country(r))
    return sorted(set(cleaned))            # <-- dedupe: presence-based counting rule

if COL_COUNTRY and COL_COUNTRY in work.columns:
    work["_countries"] = work[COL_COUNTRY].map(to_countries)
    n_geo = int((work["_countries"].map(len) > 0).sum())
    print(f"papers with >=1 resolved country: {n_geo:,} / {len(work):,}")

    ec = work.explode("_countries").explode("Diseases_canonical")
    ec = ec.dropna(subset=["_countries", "Diseases_canonical"])
    ec = ec[ec["Diseases_canonical"] != ""]
    dc = (ec.groupby(["_countries", "Diseases_canonical"]).size()
            .reset_index(name="papers"))          # each paper contributes once per (country,disease)
    dc.to_csv(f"{OUTPUT_DIR}/disease_by_country.csv", index=False)
    display(dc.sort_values("papers", ascending=False).head(20))
else:
    print("No country column set — set COL_COUNTRY in CONFIG to run 9c/9d.")

### 9d. Sub-Saharan Africa share — the headline number
Presence-based: a paper is "SSA" if **any** author is affiliated in an SSA country.
The SSA set below follows the **World Bank** region (≈48 economies). Boundary cases (Sudan, Djibouti,
Mauritania, Somalia) differ across the World Bank vs UN M49 definitions — **lock one definition and cite it**;
the number moves slightly with the choice. Denominator = papers with ≥1 resolved country.

In [ ]:
SSA_COUNTRIES = {
    # West
    "Benin","Burkina Faso","Cabo Verde","Cote d'Ivoire","Gambia","Ghana","Guinea",
    "Guinea-Bissau","Liberia","Mali","Mauritania","Niger","Nigeria","Senegal","Sierra Leone","Togo",
    # Central
    "Cameroon","Central African Republic","Chad","Congo","DR Congo","Equatorial Guinea",
    "Gabon","Sao Tome and Principe",
    # East
    "Burundi","Comoros","Djibouti","Eritrea","Ethiopia","Kenya","Madagascar","Malawi","Mauritius",
    "Rwanda","Seychelles","Somalia","South Sudan","Sudan","Tanzania","Uganda",
    # Southern
    "Angola","Botswana","Eswatini","Lesotho","Mozambique","Namibia","South Africa","Zambia","Zimbabwe",
}
DENOMINATOR = "papers_with_country"   # or "all_papers"

if COL_COUNTRY and COL_COUNTRY in work.columns:
    is_ssa = work["_countries"].map(lambda cs: any(c in SSA_COUNTRIES for c in cs))
    has_geo = work["_countries"].map(len) > 0
    denom = int(has_geo.sum()) if DENOMINATOR == "papers_with_country" else len(work)
    ssa_n = int((is_ssa & has_geo).sum())
    share = ssa_n / denom if denom else 0
    print(f"SSA papers: {ssa_n:,}")
    print(f"Denominator ({DENOMINATOR}): {denom:,}")
    print(f"==> SSA share of tick literature = {share:.1%}")

    # top SSA producing countries
    ssa_exp = work.explode("_countries")
    ssa_exp = ssa_exp[ssa_exp["_countries"].isin(SSA_COUNTRIES)]
    top_ssa = ssa_exp["_countries"].value_counts().head(15)
    top_ssa.to_csv(f"{OUTPUT_DIR}/top_ssa_countries.csv")
    plt.figure(figsize=(8,5))
    top_ssa.iloc[::-1].plot(kind="barh")
    plt.xlabel("Papers (presence-based)"); plt.title("Top SSA countries in tick literature")
    plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/top_ssa_countries.png", dpi=150); plt.show()

    # optional: SSA share *within* tick-borne-disease papers only
    tbd_mask = work["Diseases_canonical"].map(len) > 0
    if tbd_mask.any():
        d_denom = int((has_geo & tbd_mask).sum())
        d_ssa = int((is_ssa & has_geo & tbd_mask).sum())
        print(f"SSA share within TBD-labelled papers = {d_ssa}/{d_denom} = {(d_ssa/d_denom):.1%}" if d_denom else "n/a")
else:
    print("Set COL_COUNTRY in CONFIG to compute the SSA share.")

In [ ]:
top_ssa

## 10. Save the enriched record-level table

In [ ]:
keep = [c for c in [COL_ID, COL_YEAR, COL_COUNTRY, COL_TITLE, COL_ABSTRACT, KEY_WORDS, COL_LANG,"Diseases", "Diseases_canonical",
                    "TBD_gazetteer", "Tick_vectors"] if c in work.columns]
work[keep].to_parquet(f"{OUTPUT_DIR}/records_with_diseases.parquet", index=False)
work[keep].to_csv(f"{OUTPUT_DIR}/records_with_diseases.csv", index=False)
print("saved:", f"{OUTPUT_DIR}/records_with_diseases.parquet")

## 11. Outputs & next steps
**Written:** `disease_frequency`, `disease_by_year` (climate-niche), `disease_by_country` (SSA share),
`tbd_icd11_map`, `bc5cdr_disease_longtail_for_openrefine`, and the record-level `records_with_diseases.parquet`.

**To finish the disease module for the paper:**
- Verify ICD-11 stem codes (WHO API cell) — needed for a rigorous PLOS NTD methods section.
- Optionally run the OpenRefine GUI pass on the exported long tail, then fold new synonyms back into `TBD_TERMS`.
- Lock the SSA definition (World Bank vs UN M49) and cite it; §9d prints the headline share both overall and within TBD-labelled papers.
- Sanity-check `to_countries` against a sample of your real affiliation strings before trusting the number.

**Next notebook (03): collaboration & key-player networks** — NetworkX centrality on author / country
co-occurrence. It will **reuse `to_countries` unchanged** so the presence-based counting rule (and the SSA
figure) stays identical across every table and network in the paper.
